# Predicting Airbnb Listing Popularity in New York City

**Author:** Vida Saeedzadeh  
**Project:** NYC Airbnb Popularity Prediction  
**Dataset:** NYC Airbnb Open Data (2019)

---

## Table of Contents

1. [Introduction](#1-introduction)  
2. [Dataset](#2-dataset)  
3. [Project Pipeline](#3-project-pipeline)  
4. [Data Dictionary](#4-data-dictionary)  

---

## 1. Introduction

Airbnb hosts and the platform itself often want to understand what factors influence the popularity of listings. For hosts, understanding popularity drivers can help them improve listing visibility and booking rates. For the platform, predicting listing popularity can support better search ranking, host recommendations, and supply-demand balancing.

In this project, we develop a **machine learning pipeline** to predict the popularity of Airbnb listings in New York City using the **NYC Airbnb Open Data (2019)** dataset.

Because direct booking data is not available in this dataset, listing popularity is approximated using the variable **`reviews_per_month`**, which represents the average number of reviews a listing receives each month.

Although review frequency is not a perfect measure of demand, it provides a useful proxy for listing engagement and activity when booking information is unavailable.

The objectives of this project are:

- Build a **reproducible machine learning pipeline** for Airbnb listing analysis
- Explore factors associated with listing popularity
- Develop predictive models for **`reviews_per_month`**
- Evaluate model performance and interpret important features

The project is implemented using a **modular pipeline architecture**, separating data validation, cleaning, preprocessing, feature engineering, and modeling into independent steps.

---

## 2. Dataset

The dataset used in this project is the **New York City Airbnb Open Data (2019)** dataset, available on Kaggle:

https://www.kaggle.com/dgomonov/new-york-city-airbnb-open-data

The dataset contains detailed information about Airbnb listings in New York City, including listing characteristics, host information, location, pricing, availability, and review activity.

Each row in the dataset represents a single Airbnb listing.

The dataset contains approximately **48,895 listings** and **16 variables** describing listing attributes.

The raw dataset is stored in the repository at:
`data/raw/AB_NYC_2019.csv`

During the pipeline, cleaned and processed datasets are saved to:
`data/processed/`


This separation ensures that the original raw dataset remains unchanged while allowing intermediate processing steps to generate derived datasets.

---

## 3. Project Pipeline

This project follows a modular and reproducible machine learning workflow.

The pipeline is implemented using scripts in the `scripts/` directory and reusable modules in the `src/` directory.

The workflow consists of the following stages:

### 1. Data Overview / Validation

Inspect dataset structure, variable types, missing values, and potential inconsistencies.

Script:
`scripts/run_data_overview.py`


This stage produces summary tables describing:

- dataset dimensions
- column data types
- missing value statistics
- numeric constraint checks

These outputs are stored in:
`results/tables/`


---
## 2. Data Dictionary

The dataset contains the following variables describing Airbnb listings.

| Column | Description |
|------|------|
| id | Unique identifier for each Airbnb listing |
| name | Title of the listing provided by the host |
| host_id | Unique identifier for the host |
| host_name | Name of the host |
| neighbourhood_group | Borough of the listing (Manhattan, Brooklyn, Queens, Bronx, Staten Island) |
| neighbourhood | Neighborhood within the borough |
| latitude | Geographic latitude of the listing |
| longitude | Geographic longitude of the listing |
| room_type | Type of listing offered (Entire home/apt, Private room, Shared room) |
| price | Price per night in USD |
| minimum_nights | Minimum number of nights required for booking |
| number_of_reviews | Total number of reviews received by the listing |
| last_review | Date of the most recent review |
| reviews_per_month | Average number of reviews received per month |
| calculated_host_listings_count | Number of listings managed by the host |
| availability_365 | Number of days the listing is available for booking during the year |

---

### 3. Data Cleaning

Perform structural cleaning operations to prepare the dataset for analysis.

Cleaning steps include:

- Removing identifier columns that are not useful for modeling: 
  `id`, `host_id`, and `host_name`
- Removing duplicate rows
- Converting `last_review` to a datetime variable
- Removing rows with clearly invalid numeric values (e.g. `price` < 0)

Script:
`scripts/run_data_cleaning.py`

The cleaned dataset is saved to:
`data/processed/airbnb_cleaned.csv`

---

### Target Variable

The target variable used in this project is:

**`reviews_per_month`**

This variable represents the average number of reviews a listing receives each month and is used as a proxy for listing popularity.

Listings with higher values of `reviews_per_month` can be interpreted as more frequently booked or more actively used listings.

Although this measure does not directly represent bookings, it provides a useful signal of listing demand when booking data is unavailable.

---

### 4. EDA

### 4.1 Missing Values
<img src="../figures/missingness_heatmap.png" width="400">

*Figure 1: Correlation matrix of missing-value indicators showing perfect correlation between missing values in `reviews_per_month` and `last_review`.*

To better understand the structure of missing values in the dataset, we examined whether missingness in the variables is systematic. First, we computed a missingness correlation matrix, where each variable is converted to a binary indicator representing whether the value is missing.The color squares shows where values are missing. Correlations between these indicators reveal whether missing values tend to occur together. The resulting heatmap shows a perfect correlation (1.0) between missing values in reviews_per_month and last_review, indicating that these two variables are always missing simultaneously.

**Relationship between review activity and missing values**

| number_of_reviews = 0  | count | reviews_per_month_missing_rate | last_review_missing_rate |
|------------------------|------:|--------------------------------:|-------------------------:|
| FALSE                  | 38833 | 0.0 | 0.0 |
| TRUE                   | 10051 | 1.0 | 1.0 |

To further investigate this pattern, we analyzed whether missing values in these variables are associated with listings that have zero recorded reviews. The table above shows that for all listings where number_of_reviews = 0 (10,051 listings), both reviews_per_month and last_review are missing. Conversely, for listings with at least one review (38,833 listings), both variables are always present. This confirms that the missingness in these variables is structural rather than random.

This insight is important for later preprocessing steps, as it suggests that missing values in reviews_per_month should not be treated as random missing data. Instead, they represent listings with no review activity, and appropriate handling (such as imputing zero values) can be considered during the feature engineering stage.

### 4.2 Numeric Feature Distributions

<img src="../figures/numeric_histograms.png" width="800">

*Figure 2: Histograms showing the distributions of the main numeric variables, highlighting differences in scale and strong right-skewness in several features.*

As an initial exploratory step, we visualize the distributions of the numeric variables using histograms. Histograms provide a simple way to examine the shape of the data distribution and can reveal characteristics such as skewness, outliers, and concentration of values. Understanding these properties is useful for later preprocessing steps such as scaling, transformations, and feature engineering.

The figure below shows histograms for the main numeric features in the dataset.

Several observations can be made from these plots. First, the numeric variables span very different value ranges, which suggests that scaling or normalization may be useful during the modeling stage. Second, many variables exhibit strong right-skewed distributions, particularly `price`, `minimum_nights`, `number_of_reviews`, `reviews_per_month`, and `calculated_host_listings_count`. This indicates the presence of a small number of listings with very large values compared to the majority of listings.

In contrast, the geographic coordinates (`latitude` and `longitude`) display approximately bell-shaped distributions, reflecting the spatial concentration of Airbnb listings within New York City. The `availability_365` variable shows a wide spread, indicating that listings vary significantly in how frequently they are available for booking.

These distributional patterns suggest that some variables may benefit from log transformations or other preprocessing techniques in later stages of the pipeline to reduce skewness and stabilize model behavior.

### 4.3 Correlation Plot and Correlation Matrix

Before building machine learning models, it is useful to examine the relationships between numeric variables in the dataset. Correlation analysis helps identify potential dependencies between features, detect redundant variables, and understand how strongly different features are associated with the target variable. To explore these relationships, we generated two complementary visualizations: a pairwise scatter plot matrix and a correlation heatmap.

<img src="../figures/pairwise_numeric_relationships.png" width="800">

*Figure 3: Pairwise scatter plots of selected numeric features illustrating weak relationships between most variables and a moderate positive association between `number_of_reviews` and `reviews_per_month`.*

The pairwise scatter plot matrix visualizes the relationships between selected numeric variables by plotting each variable against the others. The diagonal panels show the distribution of each feature, while the off-diagonal panels show scatter plots between pairs of variables. This plot allows us to visually inspect patterns such as linear relationships, clusters, and potential outliers. From this figure we observe that most feature relationships appear weak or highly dispersed, indicating limited linear dependence between many of the variables. However, a noticeable positive relationship exists between `number_of_reviews` and `reviews_per_month`, which is expected since listings that receive more reviews overall tend to accumulate reviews more frequently over time.

<img src="../figures/correlation_matrix.png" width="400">

*Figure 4: correlation matrix of numeric features showing generally weak correlations across variables, with the strongest relationship observed between `number_of_reviews` and `reviews_per_month`.*


To quantify these relationships, we computed a correlation matrix using the Pearson correlation coefficient for the numeric variables. The heatmap provides a compact summary of pairwise linear relationships between all selected features. Consistent with the scatter plots, most correlations are relatively small in magnitude, suggesting that the numeric features capture largely distinct aspects of the listings. The strongest relationship observed is between `number_of_reviews` and `reviews_per_month` (≈ 0.55), indicating a moderate positive association between total review count and monthly review activity.

Other variables show weaker correlations with the target variable reviews_per_month. For example, `availability_365` and `calculated_host_listings_count` exhibit small positive correlations, suggesting that listings with greater availability or hosts managing more properties may receive slightly more frequent reviews. Geographic coordinates (`latitude` and `longitude`) show almost no correlation with most variables, indicating that spatial location alone does not strongly determine listing popularity in this simple linear sense.

Overall, the correlation analysis suggests that most numeric features are not strongly linearly dependent, uggesting that they capture different aspects of the listings. At the same time, the moderate relationship between review-related variables highlights an intuitive behavioral pattern in the data. Because many relationships appear nonlinear and the feature distributions are highly skewed, later modeling steps benefit from algorithms capable of capturing nonlinear interactions between variables.

---

### 5. Feature Engineering

After examining the dataset through exploratory data analysis, several transformations were applied to create more informative features and prepare the dataset for modeling. Feature engineering aims to transform the raw variables into representations that better capture meaningful patterns in the data.

The following feature engineering steps were implemented.

### 5.1 Log Transformation of Skewed Variables

The histograms of the numeric variables in EDA showed strong right-skewed distributions for several features, including:
`price`, `minimum_nights`, `number_of_reviews`, `reviews_per_month`, and `calculated_host_listings_count`

Highly skewed variables can negatively affect certain machine learning algorithms and may obscure relationships between variables. To mitigate this issue, we applied a logarithmic transformation using the function $log(1+x)$ .

This transformation compresses large values while preserving the ordering of observations, producing more stable distributions. The transformed variables were added as new features: `log_price`, `log_minimum_nights` etc. 

The original variables were retained to allow later models to determine which representation provides the most predictive signal.

### 5.2 Spatial Feature: Distance from Manhattan

The dataset includes geographic coordinates (latitude and longitude) for each listing. While these coordinates contain spatial information, they are not directly interpretable for many models.
To create a more meaningful spatial feature, we computed the **distance of each listing from the center of Manhattan**. Manhattan serves as a useful reference point because it represents the primary economic and tourism center of New York City.

For each listing, the Euclidean distance from this location was calculated and converted to kilometers. The resulting feature: `distance_from_manhattan_km`. captures how far a listing is located from the city center, which may influence its attractiveness and review activity.

After generating this feature, the original coordinate variables (`latitude` and `longitude`) were removed to avoid redundant information.


### 5.3 Sentiment Analysis of Listing Names

Listing names often contain descriptive language intended to attract potential guests. According to the [rental tips](https://www.igms.com/airbnb-titles/), a well-crafted Airbnb title can attract up to 5X more bookings. To extract information from this text field, we applied sentiment analysis to the name column using the VADER (Valence Aware Dictionary and Sentiment Reasoner) sentiment model from the NLTK library.

For each listing title, VADER produces a compound sentiment score ranging from:

- −1 (very negative sentiment)

- 0 (neutral sentiment)

- +1 (very positive sentiment)

This score provides a numeric measure of how positive or negative the listing description appears. The resulting feature: `name_sentiment` was added to the dataset to capture potential differences in how listings are presented by hosts.

After extracting the sentiment score, the original text column (`name`) was removed from the dataset.

### 5.4 Review Recency

The variable last_review contains the date of the most recent review for each listing. Since raw dates cannot be used directly in most machine learning models, this information was converted into a numeric feature called `days_since_last_review`. This feature measures the number of days between the listing’s most recent review and the latest review date observed in the dataset. Smaller values indicate recently active listings, while larger values indicate listings that have not received reviews for a longer period. After creating this feature, the original last_review column was removed.


### 5.5 Host Professionalism

The variable `calculated_host_listings_count` indicates how many listings a host manages. To capture whether a host operates multiple properties, we created a binary feature called `is_professional_host`. Listings were labeled as 1 if the host manages more than one listings and 0 otherwise. This feature allows the model to distinguish between individual hosts and hosts managing multiple properties, which may reflect differences in listing management and booking activity.

### 5.6 Removing Target-Derived Information

The variable number_of_reviews is moderately correlated with the target variable reviews_per_month, since listings that accumulate more reviews tend to receive reviews more frequently. However, this variable contains information that is closely related to the target and could allow the model to partially infer the outcome rather than learn from independent listing characteristics.

More importantly, the goal of this project is to predict the popularity of new listings before they appear on the platform. At that stage, a listing has not yet received any reviews, so number_of_reviews would not be available.

To ensure the model remains realistic and generalizable, the number_of_reviews column was removed from the feature set.

---

### 6. Preprocessing and Transformations

Preprocessing in machine learning refers to the steps taken to prepare data for use in a model. This can include cleaning and formatting the data, as well as normalizing or scaling feature values. It is an essential stage in building a stable machine learning pipeline, since these steps help improve the performance and reliability of a model and can also reduce the risk of overfitting.

### 6.1 Handling Missing Values in the Target Variable

Exploratory data analysis showed that missing values in the target variable reviews_per_month occur when a listing has zero reviews. In this situation, the absence of reviews indicates that the listing has not yet received any review activity rather than representing random missing data.

Therefore, missing values in `reviews_per_month` were replaced with 0, reflecting the fact that these listings have not yet accumulated monthly review activity.

### 6.2 Train–Test Split

To ensure that model evaluation reflects the ability to generalize to unseen data, the dataset was divided into training and testing sets. The split was performed using the train_test_split function from Scikit-learn.

The dataset was partitioned using:

- 80% of the data for training

- 20% of the data for testing

The training set is used to fit the preprocessing transformations and train machine learning models, while the testing set remains untouched until the evaluation stage.


### 6.3 Numeric Feature Processing

Numeric features were processed using a preprocessing pipeline consisting of two steps:

1. Median Imputation

Missing values in numeric features were replaced using the median value of the corresponding feature in the training set. Median imputation is robust to outliers and is commonly used for skewed data distributions.

2. Feature Scaling

Numeric variables were standardized using StandardScaler, which transforms each feature to have: mean = 0 and standard deviation = 1.

Scaling ensures that features with larger numeric ranges do not dominate the learning process for algorithms sensitive to feature scale.


### 6.4 Categorical Feature Encoding

Categorical variables were converted into numeric representations using One-Hot Encoding.

One-hot encoding creates a binary indicator variable for each category level, allowing machine learning models to interpret categorical features without introducing artificial ordinal relationships.

The following categorical variables were encoded:

- `neighbourhood_group`

- `room_type`

Missing categorical values were first replaced with the label "missing" before encoding.


### 6.5 Feature Selection

The variable `neighbourhood` was removed during preprocessing. Although it contains location information, the column has a large number of unique values, which would lead to a very high number of encoded features and increase model complexity. Instead, location effects are captured through the engineered feature `distance_from_manhattan_km` and the borough-level variable `neighbourhood_group`.


### 6.6 Preprocessing Pipeline

All preprocessing transformations were implemented using a Scikit-learn `ColumnTransformer` pipeline. This approach ensures that preprocessing steps are applied consistently during both training and prediction.

The pipeline performs the following operations:

- median imputation and scaling for numeric variables

- imputation and one-hot encoding for categorical variables

- removal of unused columns

Fitting the preprocessing pipeline on the training data ensures that statistics such as imputation values and scaling parameters are learned only from the training set, preventing information leakage from the test data.

The fitted preprocessing pipeline was saved to disk so that the exact transformations can be reused during model training and inference.

### 6.7 Final Modeling Dataset

After preprocessing, the dataset consists entirely of numeric features ready for model training. The transformed feature matrices for the training and testing sets were saved for use in the modeling stage.

This structured preprocessing pipeline ensures that the dataset is consistently prepared and that the modeling process remains reproducible.
